# Semantic transfer and routing evidence

The transfer study changes task compositions and preserves its own lineage. The routing study reports a fitted allocation gate separately from the task outcomes. This notebook reads both source-bound reports and compares selected methods without pooling transfer with the core matrix.

## Recompute the source-bound summaries

The following cell reads committed evidence only. It does not execute a policy or create new results.

In [1]:
from pathlib import Path
import hashlib, json, os
ROOT = Path(os.environ['NEURAXIS_REPO_ROOT'])
data = ROOT / 'data' / 'derived'
core = json.loads((data / 'index.json').read_text(encoding='utf-8'))
transfer_path = data / 'semantic-transfer' / 'index.json'
transfer_bytes = transfer_path.read_bytes()
transfer = json.loads(transfer_bytes)
report = json.loads((data / 'semantic-transfer' / 'report.json').read_text(encoding='utf-8'))
route = json.loads((data / 'semantic-routing-study' / 'analysis.json').read_text(encoding='utf-8'))
assert hashlib.sha256(transfer_bytes).hexdigest() == report['index_sha256']
metrics = {row['method_id']: row for row in report['methods']}
core_metrics = {row['method_id']: row for row in core['aggregate_results']}
print('Core and semantic transfer are separate datasets; do not pool their denominators.')
print('Transfer episodes:', report['episodes_checked'], '| families:', len(transfer['cases']))
print(f"{'Method':<9} {'Transfer successes':>20} {'Episodes':>10} {'Rate':>8} {'Mean wall ms':>14}")
for method in ('M04', 'M09', 'M12'):
    row = metrics[method]
    print(f"{method:<9} {row['successes']:>20} {row['episodes']:>10} {row['successes']/row['episodes']:>8.3f} {row['mean_wall_ms']:>14.2f}")
print('M12 paired outcomes:', json.dumps(report['m12_paired'], sort_keys=True))
print('Routing gate threshold:', route['selected_threshold'], '| fit positives:', route['fit_positives'], '| calibration positives:', route['calibration_positives'])
for label, result in route['test'].items():
    print(f"routing {label}: episodes={result['episodes']} verified_successes={result['verified_successes']} planner_episodes={result['planner_episodes']} mean_wall_ms={result['mean_wall_ms']:.2f}")
print('Core M04 and M12 rates:', f"{core_metrics['M04']['task_success']:.3f}", f"{core_metrics['M12']['task_success']:.3f}")

Core and semantic transfer are separate datasets; do not pool their denominators.
Transfer episodes: 1536 | families: 8
Method      Transfer successes   Episodes     Rate   Mean wall ms
M04                          2        128    0.016         329.22
M09                         38        128    0.297       18882.47
M12                         18        128    0.141        7706.12
M12 paired outcomes: {"M01": {"m12_wins": 18, "ties": 110}, "M04": {"m12_losses": 2, "m12_wins": 18, "ties": 108}, "M09": {"m12_losses": 22, "m12_wins": 2, "ties": 104}}
Routing gate threshold: 0.44629505030857797 | fit positives: 19 | calibration positives: 8
routing always_fast_M04: episodes=32 verified_successes=0 planner_episodes=0 mean_wall_ms=391.45
routing always_planner_M09: episodes=32 verified_successes=10 planner_episodes=32 mean_wall_ms=20052.69
routing hash_half: episodes=32 verified_successes=7 planner_episodes=19 mean_wall_ms=11568.07
routing learned_gate: episodes=32 verified_successes=0 plann

## Interpretation limits

Held-out transfer and the small routing sample remain bounded evidence. A positive paired count is not a proof of broad transfer; the threshold, planner cost, invalid outputs and denominator must be read together.